# Chapter 5: Optimization

*Companion notebook for* **The Math That Powers AI** *(2nd edition)*.

This notebook reproduces the chapter's worked examples using the
`mathpowersai.optimizers` package:

1. Gradient descent on the book's quadratic $f(x, y) = x^2 + 4y^2$
2. The learning-rate divergence threshold $\eta_{\max} = 2 / \lambda_{\max}$
3. The condition-number convergence rate at the optimal step size
4. Comparing optimizers with their per-optimizer default learning rates
5. Reproducible stochastic gradient descent with a seeded generator

All cells are deterministic and run offline.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np

## 1. The book's worked example: $f(x, y) = x^2 + 4y^2$

Starting from $(2, 1)$ with learning rate $\eta = 0.1$, the gradient is
$\nabla f = (2x, 8y)$, so the first step lands at

$$(2, 1) - 0.1 \cdot (4, 8) = (1.6, 0.2).$$

Each coordinate contracts geometrically ($x$ by $0.8$ per step, $y$ by $0.2$),
so after 10 iterations we expect roughly $(0.21,\ 10^{-7})$ with
$f \approx 0.046$ — exactly the table in the book.

In [ ]:
from mathpowersai.optimizers import gradient_descent


def f(p):
    return p[0] ** 2 + 4 * p[1] ** 2


def grad_f(p):
    return np.array([2 * p[0], 8 * p[1]])


path = gradient_descent(grad_f, np.array([2.0, 1.0]), lr=0.1, n_steps=10)

print(f"Start:   ({path[0][0]:.4g}, {path[0][1]:.4g}),  f = {f(path[0]):.2f}")
print(f"Step 1:  ({path[1][0]:.4g}, {path[1][1]:.4g}),  f = {f(path[1]):.2f}")
print(f"Step 10: ({path[-1][0]:.2f}, {path[-1][1]:.1e}),  f = {f(path[-1]):.3f}")

# Match the book's numbers.
assert np.allclose(path[1], [1.6, 0.2])
assert np.isclose(path[-1][0], 2.0 * 0.8 ** 10)      # ~0.2147
assert np.isclose(path[-1][1], 1.0 * 0.2 ** 10)      # ~1.0e-07
assert np.isclose(f(path[-1]), 0.046, atol=1e-3)

## 2. The divergence threshold: $\eta_{\max} = 2 / \lambda_{\max}$

For a quadratic, gradient descent is stable only while every coordinate's
update factor $|1 - \eta \lambda_i|$ stays below 1. The binding constraint is
the largest Hessian eigenvalue. Here the Hessian is
$\operatorname{diag}(2, 8)$, so $\lambda_{\max} = 8$ and

$$\eta_{\max} = \frac{2}{\lambda_{\max}} = \frac{2}{8} = 0.25.$$

A learning rate of $0.09$ sits safely below the threshold and converges;
$0.26$ sits just above it, making the $y$-coordinate factor
$|1 - 0.26 \cdot 8| = 1.08 > 1$, so the iterates oscillate and blow up.

In [ ]:
lambda_max = 8.0
eta_max = 2.0 / lambda_max
print(f"Divergence threshold: eta_max = 2 / {lambda_max:.0f} = {eta_max}")

x0 = np.array([2.0, 1.0])
path_stable = gradient_descent(grad_f, x0, lr=0.09, n_steps=50)
path_diverge = gradient_descent(grad_f, x0, lr=0.26, n_steps=50)

print(f"lr = 0.09 (stable):    f after 50 steps = {f(path_stable[-1]):.3e}")
print(f"lr = 0.26 (divergent): f after 50 steps = {f(path_diverge[-1]):.3e}")
print(f"lr = 0.26 y-iterates oscillate: {[round(float(p[1]), 3) for p in path_diverge[:5]]}")

assert f(path_stable[-1]) < 1e-3          # well below the start, f(x0) = 8
assert f(path_diverge[-1]) > 1e3          # blown far past the start

## 3. Condition number and the optimal step size

The condition number of the Hessian is
$\kappa = \lambda_{\max} / \lambda_{\min} = 8 / 2 = 4$. The step size that
balances the fastest and slowest directions is

$$\eta^* = \frac{2}{\lambda_{\min} + \lambda_{\max}} = \frac{2}{10} = 0.2,$$

which yields the classic per-step contraction rate

$$\rho = \frac{\kappa - 1}{\kappa + 1} = \frac{3}{5} = 0.6.$$

At $\eta^*$ both coordinate factors equal $0.6$ in magnitude
($1 - 0.2 \cdot 2 = 0.6$ and $1 - 0.2 \cdot 8 = -0.6$), so the distance to
the optimum shrinks by exactly $\rho$ every iteration.

In [ ]:
lambda_min, lambda_max = 2.0, 8.0
kappa = lambda_max / lambda_min
eta_opt = 2.0 / (lambda_min + lambda_max)
rho = (kappa - 1.0) / (kappa + 1.0)
print(f"kappa = {kappa},  optimal lr = {eta_opt},  predicted rate = {rho}")

path_opt = gradient_descent(grad_f, np.array([2.0, 1.0]), lr=eta_opt,
                            n_steps=20)

dists = [np.linalg.norm(p) for p in path_opt]  # optimum is the origin
ratios = [dists[k + 1] / dists[k] for k in range(5)]
print("Per-step contraction ||x_{k+1}|| / ||x_k||:",
      [round(r, 6) for r in ratios])

assert np.allclose(ratios, rho)
assert np.isclose(dists[10], dists[0] * rho ** 10)

## 4. Comparing optimizers (per-optimizer default learning rates)

`compare_optimizers` runs several optimizers on the same problem. With
`lr=None` (the default) each optimizer keeps its own documented default —
$0.01$ for gradient descent, momentum, and RMSprop, and $0.001$ for Adam —
rather than forcing one shared step size on all of them.

In [ ]:
from mathpowersai.optimizers import compare_optimizers

results = compare_optimizers(grad_f, np.array([2.0, 1.0]), n_steps=50)

print("Final loss after 50 steps (per-optimizer default lr):")
for name, p in results.items():
    print(f"  {name:18s} f = {f(p[-1]):.6f}")

assert set(results) == {'gradient_descent', 'momentum', 'rmsprop', 'adam'}
# Every optimizer makes progress from f(x0) = 8.
assert all(f(p[-1]) < f(p[0]) for p in results.values())

## 5. Reproducible SGD with a seeded generator

`sgd` simulates mini-batch noise by adding Gaussian perturbations to the
gradient. Passing a seeded `np.random.Generator` makes the entire path
reproducible: two runs with the same seed are bit-for-bit identical, while a
different seed gives a different (but equally valid) path.

In [ ]:
from mathpowersai.optimizers import sgd

x0 = np.array([2.0, 1.0])
path_a = sgd(grad_f, x0, lr=0.05, n_steps=100, noise_scale=0.1,
             rng=np.random.default_rng(42))
path_b = sgd(grad_f, x0, lr=0.05, n_steps=100, noise_scale=0.1,
             rng=np.random.default_rng(42))
path_c = sgd(grad_f, x0, lr=0.05, n_steps=100, noise_scale=0.1,
             rng=np.random.default_rng(7))

same = all(np.array_equal(a, b) for a, b in zip(path_a, path_b))
differ = any(not np.array_equal(a, c) for a, c in zip(path_a, path_c))
print(f"Seed 42 vs seed 42: identical paths = {same}")
print(f"Seed 42 vs seed 7:  paths differ    = {differ}")
print(f"Seed-42 final point: ({path_a[-1][0]:.6f}, {path_a[-1][1]:.6f}),"
      f" f = {f(path_a[-1]):.6f}")

assert same and differ

## Optional: convergence plot

This last cell visualizes the loss curves from Section 4 with
`mathpowersai.visualization.plot_convergence`. It is guarded so the notebook
still runs headless (Agg backend, no `plt.show()`) and is skipped entirely if
matplotlib is not installed.

In [ ]:
try:
    import matplotlib
    matplotlib.use("Agg")  # headless backend; must precede any pyplot import
    from mathpowersai.visualization import plot_convergence
except ImportError as exc:
    print(f"matplotlib unavailable; skipping plot ({exc})")
else:
    ax = plot_convergence(results, f,
                          title="Chapter 5: optimizer convergence on f(x, y) = x^2 + 4y^2")
    print("Rendered convergence plot:", ax.get_title())